# EXAONE 4.0 1.2B QLoRA 파인튜닝 (v2, Colab T4용)

`labeled_dataset.jsonl`(Gemini 라벨링 결과, 1999건)을 `질문+답변 → 피드백` instruction 데이터로 변환해서
EXAONE 4.0 1.2B를 QLoRA로 파인튜닝.

**학습 목표는 4개 항목(두괄식/논리구조/키워드/분량) 채점 및 피드백 생성만** — 개선답변/꼬리질문은
포함하지 않음 (개선답변은 계획에 없던 기능, 꼬리질문은 1팀 Gemini 파트 담당).

## 1. 패키지 설치

In [4]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets

## 2. Google Drive 마운트

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. 설정값

In [6]:
LABELED_PATH = "/content/drive/MyDrive/Colab Notebooks/AIM/AIHub_면접데이터/labeled_dataset.jsonl"        # 라벨링 완료된 데이터 경로로 수정
SAVE_DIR = "/content/drive/MyDrive/exaone_qlora_v2"     # 학습된 어댑터 저장 경로 (Drive)
TEST_SET_PATH = "/content/drive/MyDrive/Colab Notebooks/AIM/exaone_qlora_v1/test_set.jsonl"  # 비교실험용 test set 저장 경로
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"

MAX_LENGTH = 1024
# NUM_EPOCHS = 3
NUM_EPOCHS = 5
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4    # 실질 배치 크기 = 16
LEARNING_RATE = 2e-4

## 4. 라벨 → 피드백 텍스트 변환
기획서 피드백 출력 형식(✅/❌/⚠️)에 맞춰 라벨을 사람이 읽는 텍스트로 변환. **4개 항목(두괄식/논리구조/키워드/분량)만 다룸 — 개선답변/꼬리질문은 포함하지 않음**

In [7]:
import json

def labels_to_feedback_text(labels: dict) -> str:
    lines = []

    if labels["headline"]:
        lines.append("[두괄식] ✅ 결론이 첫 문장에 명확히 드러납니다.")
    else:
        lines.append('[두괄식] ❌ 결론 없이 시작됩니다.\n → "저는 ~한 경험으로 ~을 배웠습니다"로 시작해보세요.')

    score = labels["logic_structure"]
    reason = labels["logic_reason"]
    mark = "✅" if score >= 4 else ("⚠️" if score == 3 else "❌")
    lines.append(f"[논리 구조] {mark} {reason} (점수: {score}/5)")

    missing = labels["missing_keywords"]
    if missing:
        missing_str = ", ".join(f"'{k}'" for k in missing)
        lines.append(f"[키워드] ⚠️ {missing_str} 키워드가 빠져있습니다.")
    else:
        lines.append("[키워드] ✅ 직무 관련 핵심 키워드가 잘 포함되어 있습니다.")

    char_count = labels["char_count"]
    if labels["is_appropriate"]:
        lines.append(f"[분량] ✅ 적정 분량입니다. ({char_count}자)")
    else:
        lines.append(f"[분량] ⚠️ 분량이 적정 범위(150~400자)를 벗어났습니다. ({char_count}자)")

    return "\n".join(lines)


sample_labels = {
    "headline": False, "logic_structure": 2, "logic_reason": "서론이 길고 결론이 모호함",
    "found_keywords": [], "missing_keywords": ["협업", "성과"],
    "char_count": 230, "is_appropriate": True,
}
print(labels_to_feedback_text(sample_labels))

[두괄식] ❌ 결론 없이 시작됩니다.
 → "저는 ~한 경험으로 ~을 배웠습니다"로 시작해보세요.
[논리 구조] ❌ 서론이 길고 결론이 모호함 (점수: 2/5)
[키워드] ⚠️ '협업', '성과' 키워드가 빠져있습니다.
[분량] ✅ 적정 분량입니다. (230자)


## 5. Instruction 데이터셋 구성 (messages 포맷)

In [8]:
with open(LABELED_PATH, "r", encoding="utf-8") as f:
    labeled_rows = [json.loads(line) for line in f]

print(f"전체 라벨링 데이터: {len(labeled_rows)}건")
print("labels 필드:", list(labeled_rows[0]["labels"].keys()))  # 실제 필드명 확인용

SYSTEM_PROMPT = (
    "너는 대기업 전문 채용 면접관이자 AI 취업 코치이다.\n"
    "제시된 [면접 질문]과 [사용자 답변]을 분석하여 아래 4가지 항목을 평가하라.\n\n"
    "1. 두괄식: 답변이 결론부터 시작하는지 true/false로 평가하라.\n"
    "2. 논리구조: 답변이 '결론 -> 근거 -> 마무리'의 논리적 흐름을 갖추고 있는지 1~5점으로 평가하고, 그 이유를 설명하라.\n"
    "3. 키워드: 답변에서 발견된 키워드와 누락된 키워드를 각각 나열하라.\n"
    "4. 분량: 답변 글자수를 세고, 150~400자 기준으로 적정한지 평가하라.\n\n"
    "반드시 아래의 JSON 포맷으로만 응답하며, JSON 외의 서론이나 설명은 절대 추가하지 마라.\n"
    "{\n"
    '  "headline": true 또는 false,\n'
    '  "logic_structure": 1~5 사이 정수,\n'
    '  "logic_reason": "논리구조 평가 이유",\n'
    '  "found_keywords": ["발견된 키워드"],\n'
    '  "missing_keywords": ["누락된 키워드"],\n'
    '  "char_count": 정수,\n'
    '  "is_appropriate": true 또는 false\n'
    "}"
)

def build_training_example(row):
    labels = row["labels"]
    user_message = f"[면접 질문]: {row['question']}\n[사용자 답변]: {row['answer']}"

    target_json = {
        "headline": labels.get("headline"),
        "logic_structure": labels.get("logic_structure"),
        "logic_reason": labels.get("logic_reason", ""),
        "found_keywords": labels.get("found_keywords", []),
        "missing_keywords": labels.get("missing_keywords", []),
        "char_count": labels.get("char_count"),
        "is_appropriate": labels.get("is_appropriate")
    }

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": json.dumps(target_json, ensure_ascii=False)}
    ]
    return {"messages": messages}

training_examples = [build_training_example(r) for r in labeled_rows]
print(f"학습 예시 변환 완료: {len(training_examples)}건")

전체 라벨링 데이터: 1999건
labels 필드: ['headline', 'logic_structure', 'logic_reason', 'found_keywords', 'missing_keywords', 'char_count', 'is_appropriate']
학습 예시 변환 완료: 1999건


## 6. Train/Val/Test 분할 (80/10/10)
테스트셋은 이후 base 모델·GPT-4o-mini와의 비교실험에 그대로 재사용

In [9]:
import random
from datasets import Dataset
from pathlib import Path

random.seed(42)
random.shuffle(training_examples)

n = len(training_examples)
train_size = int(n * 0.8)
val_size = int(n * 0.1)

train_data = training_examples[:train_size]
val_data = training_examples[train_size:train_size + val_size]
test_data = training_examples[train_size + val_size:]

print(f"Train: {len(train_data)} / Val: {len(val_data)} / Test: {len(test_data)}")

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

# test set은 비교실험에서 재사용할 수 있게 별도 저장
Path(TEST_SET_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(TEST_SET_PATH, "w", encoding="utf-8") as f:
    for ex in test_data:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print(f"test set 저장 완료 -> {TEST_SET_PATH}")

Train: 1599 / Val: 199 / Test: 201
test set 저장 완료 -> /content/drive/MyDrive/Colab Notebooks/AIM/exaone_qlora_v1/test_set.jsonl


## 7. 모델 & 토크나이저 로드 (4bit 양자화, fp16)

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4(Turing)는 fp16이 bf16보다 안정적
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.use_cache = False
print("모델 로드 완료")

# 실제 레이어 이름 확인 (target_modules가 안 맞으면 아래 셀에서 수정)
print(model)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/70.3k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.56GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

모델 로드 완료
Exaone4ForCausalLM(
  (model): Exaone4Model(
    (embed_tokens): Embedding(102400, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-29): 30 x Exaone4DecoderLayer(
        (self_attn): Exaone4Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (q_norm): Exaone4RMSNorm((64,), eps=1e-05)
          (k_norm): Exaone4RMSNorm((64,), eps=1e-05)
        )
        (mlp): Exaone4MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=4096, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=4096, bias=False)
          (down_proj): Linear4bit(in_features=4096, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (post_attention_lay

## 8. LoRA 설정
위 셀 `print(model)` 결과 보고 `target_modules`가 실제 존재하는 레이어명인지 확인 후 진행

In [11]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # 위 print(model) 결과와 다르면 수정
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 6,389,760 || all params: 1,285,781,248 || trainable%: 0.4970


## 9. 학습 실행 (SFTTrainer, chat template 사용)

In [12]:
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/Colab Notebooks/AIM/exaone_qlora_v2_ckpt",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    bf16=False,
    max_length=MAX_LENGTH,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    formatting_func=formatting_func,
)

# trainer.train()
trainer.train(resume_from_checkpoint="/content/drive/MyDrive/Colab Notebooks/AIM/exaone_qlora_v2_ckpt/checkpoint-300")

Applying formatting function to train dataset:   0%|          | 0/1599 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1599 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1599 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1599 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1599 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/199 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/199 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/199 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
4,1.087695,1.138128,1.188068,1038713.000000,0.763832
5,1.058925,1.135940,1.160022,2077426.000000,0.765326


TrainOutput(global_step=500, training_loss=0.43553459548950196, metrics={'train_runtime': 3810.0963, 'train_samples_per_second': 2.098, 'train_steps_per_second': 0.131, 'total_flos': 3.738305101922304e+16, 'train_loss': 0.43553459548950196, 'epoch': 5.0})

## 10. 어댑터 저장 (Drive)

In [13]:
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"파인튜닝 완료, 저장 위치: {SAVE_DIR}")

파인튜닝 완료, 저장 위치: /content/drive/MyDrive/exaone_qlora_v2


## 11. 간단 추론 테스트

In [14]:
test_example = test_data[0]
prompt_messages = test_example["messages"][:-1]  # system+user만 (assistant 정답은 제외)

prompt_text = tokenizer.apply_chat_template(
    prompt_messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt_text, return_tensors="pt", return_dict=True).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== 질문/답변 ===")
print(test_example["messages"][1]["content"])
print()
print("=== 모델 생성 피드백 ===")
print(generated)
print()
print("=== 정답(Gemini 라벨 기반) 피드백 ===")
print(test_example["messages"][-1]["content"])

=== 질문/답변 ===
[면접 질문]: 혹시 지원자님께서는 지원자님의 의견을 주장하실 때 강하게 주장하는 편이십니까 지원자님께서는 어떤 스타일로 자신의 의견을 주장하시는지 말씀해 주시기 바랍니다
[사용자 답변]: 저는 자기 주장이 강한 편입니다. 현대사회는 자기 주장이 강한 사람이 성공할 확률이 높습니다. 조용하고 착한 것이 미덕인 세상은 이미 지나갔다고 생각됩니다. 이미 철저한 경쟁 사회가 되었기 때문에 제가 원하는 것이 있다면 능동적으로 적극적으로 나의 것으로 당 달성할 수 있도록 노력해야 할 것입니다. 제가 아무런 노력도 하지 않고 그냥 좋은 결과만을 기다리는 것은 가장 위험한 마음에 선택인 것 같습니다. 하지만 저의 주장만 강하게 주장하지는 않을 것입니다. 타인을 존중하고 매 배려하는 마음이 없다면 이것은 자기 주장이 강한 것이 아니라 자기 생각 속에 빠져서 내가 정답이라는 가장 독선적인 모습일 뿐이라고 생각됩니다. 그렇기 때문에 저는 주변의 타인의 의견을 수용하고 충분히 듣는 상태에서 저의 생각과 의견을 말하는 편입니다.

=== 모델 생성 피드백 ===
{"headline": true, "logic_structure": 3, "logic_reason": "결론(강한 주장)과 그에 대한 근거(경쟁 사회의 특성)가 제시되었으나, 이후 '타인 존중'이라는 가치를 강조하며 논리가 다소 산만해짐", "found_keywords": ["자기 주장", "경쟁 사회", "적극적 노력"], "missing_keywords": ["커뮤니케이션", "협업", "데이터 기반 주장"], "char_count": 338, "is_appropriate": true}

=== 정답(Gemini 라벨 기반) 피드백 ===
{"headline": true, "logic_structure": 3, "logic_reason": "자기 주장이 강하다는 결론으로 시작하여 논리적 근거를 제시하려 했으나, 초반부의 일반론적 주장과 후반부의 경청 태도가 다소 상충되어 일관성이 부족함", "f

## 12. HF Hub에 어댑터 업로드

In [16]:
from google.colab import userdata
HF_TOKEN_WRITE = userdata.get('HF_TOKEN1')

In [17]:
from huggingface_hub import login

# HF 토큰 필요 - huggingface.co/settings/tokens 에서 발급 (write 권한)
login(token=HF_TOKEN_WRITE)

REPO_ID = "shk776/exaone-interview-adapter-v2"

# 방법 1: 이미 메모리에 있는 model/tokenizer로 바로 push
model.push_to_hub(REPO_ID, private=True)
tokenizer.push_to_hub(REPO_ID, private=True)

print(f"업로드 완료: https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|2         |  564kB / 25.6MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

업로드 완료: https://huggingface.co/shk776/exaone-interview-adapter-v2
